In [ ]:
%pip install -q git+https://github.com/zhouy185/rosa-starter.git

In [2]:
import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

times = delivery_times("Far West", "Fri/Sat eve", 45, seed=1)
print(len(times), "orders; average delivery time", round(times.mean(), 1), "minutes")

210 orders; average delivery time 42.9 minutes


## Part I. Describe: How bad is it, really?

### (a) Percentage of late orders

An order is late when its delivery time is longer than the promised delivery time.

The function below takes a zone, a time block and a promise. If the zone is 'all' it looks at every zone, otherwise just the one given, and the same goes for the time block. This one rule covers all four cases in the question. For each zone and time block pair it includes, it gets the delivery times from the simulator (with seed=1 so the numbers stay the same every run), counts the orders and the late orders, and adds them up. The result is late orders divided by total orders, times 100.

I add up the orders first instead of averaging the percentages of each pair, because the pairs have different numbers of orders. A busy pair should count for more than a quiet one.

In [3]:
def late_percentage(zone, time_block, promise=PROMISE, seed=1):
    """Percentage of orders delivered later than the promise.

    zone and time_block can each be a specific name or 'all'.
    """
    zones = ZONES if zone == 'all' else [zone]
    blocks = TIME_BLOCKS if time_block == 'all' else [time_block]

    total_orders = 0
    late_orders = 0
    for z in zones:
        for b in blocks:
            times = delivery_times(z, b, promise, seed=seed)
            total_orders += len(times)
            late_orders += np.sum(times > promise)

    return 100 * late_orders / total_orders

Testing the function with the 45-minute promise, one example for each case: Far West and Fri/Sat eve, all zones at Lunch, Central across all time blocks, and everything combined.

In [4]:
test_cases = [
    ("Far West", "Fri/Sat eve"),
    ('all', "Lunch"),
    ("Central", 'all'),
    ('all', 'all'),
]

for zone, block in test_cases:
    pct = late_percentage(zone, block, 45)
    print(f"Zone: {zone:<9} | Time block: {block:<12} | Late: {pct:.1f}%")

Zone: Far West  | Time block: Fri/Sat eve  | Late: 36.7%
Zone: all       | Time block: Lunch        | Late: 3.3%
Zone: Central   | Time block: all          | Late: 4.2%
Zone: all       | Time block: all          | Late: 8.3%


Overall about 8% of orders are late, but this is not spread evenly. In the Far West on Friday/Saturday evenings about 37% of orders miss the 45-minute promise, while Lunch across all zones (about 3%) and the Central zone overall (about 4%) are well below the average. This matches what we saw in class: the lateness is concentrated in the Far West on weekends.

### (b) Ranking zone and time block pairs by late rate

Next I want to see which pairs cause the most late orders under the 45-minute promise. I loop over all 3 zones and 4 time blocks (12 pairs), get each pair's late rate with the function from part (a), store the results as tuples in a list, and sort the list from highest to lowest late rate.

In [5]:
pair_rates = []
for zone in ZONES:
    for block in TIME_BLOCKS:
        rate = late_percentage(zone, block, PROMISE)
        pair_rates.append((zone, block, rate))

ranked = sorted(pair_rates, key=lambda pair: pair[2], reverse=True)

print(f"{'Rank':<5} {'Zone':<9} {'Time block':<12} Late rate")
for rank, (zone, block, rate) in enumerate(ranked, start=1):
    print(f"{rank:<5} {zone:<9} {block:<12} {rate:5.1f}%")

Rank  Zone      Time block   Late rate
1     Far West  Fri/Sat eve   36.7%
2     North     Fri/Sat eve   16.7%
3     Far West  Weekday eve   11.2%
4     Central   Fri/Sat eve    6.8%
5     North     Weekday eve    6.0%
6     Far West  Other          4.7%
7     Central   Weekday eve    4.2%
8     North     Lunch          3.7%
9     Central   Lunch          3.5%
10    Central   Other          2.6%
11    North     Other          2.5%
12    Far West  Lunch          2.5%


Friday/Saturday evening is the worst time block in every zone. Far West on Fri/Sat evenings is by far the worst pair at about 37% late, more than double the next one (North, Fri/Sat eve, about 17%). Far West on weekday evenings is third at about 11%. Every other pair is below 7%, and the lunch and "Other" pairs are only around 2.5 to 3.5%. So the problem is concentrated in a few busy evening pairs rather than across the whole business, which suggests one promise for every zone and time block may not be the best choice.

### (c) Average delivery time

This function works the same way as the one in part (a) and handles the same four cases. Instead of counting late orders, it adds up the delivery times of all included orders and divides by the number of orders. As before, the orders are pooled across pairs rather than averaging the averages, so busier pairs count for more.

In [6]:
def average_delivery_time(zone, time_block, promise=PROMISE, seed=1):
    """Average delivery time in minutes over all included orders.

    zone and time_block can each be a specific name or 'all'.
    """
    zones = ZONES if zone == 'all' else [zone]
    blocks = TIME_BLOCKS if time_block == 'all' else [time_block]

    total_minutes = 0
    total_orders = 0
    for z in zones:
        for b in blocks:
            times = delivery_times(z, b, promise, seed=seed)
            total_minutes += np.sum(times)
            total_orders += len(times)

    return total_minutes / total_orders

Testing with the same four examples as in part (a):

In [7]:
for zone, block in test_cases:
    avg = average_delivery_time(zone, block, 45)
    print(f"Zone: {zone:<9} | Time block: {block:<12} | Average: {avg:.1f} minutes")

Zone: Far West  | Time block: Fri/Sat eve  | Average: 42.9 minutes
Zone: all       | Time block: Lunch        | Average: 28.2 minutes
Zone: Central   | Time block: all          | Average: 28.5 minutes
Zone: all       | Time block: all          | Average: 31.2 minutes


The overall average delivery takes about 31 minutes, well inside the 45-minute promise, and Lunch and Central both average about 28 minutes. Even Far West on Fri/Sat evenings averages about 43 minutes, which is still under the promise, even though part (a) showed that about 37% of its orders are late. So the average can look fine while a large share of individual orders miss the promise. I come back to this in part (e).

### (d) Ranking zone and time block pairs by average delivery time

Same steps as part (b), but ranking the 12 pairs by average delivery time at the 45-minute promise.

In [8]:
pair_averages = []
for zone in ZONES:
    for block in TIME_BLOCKS:
        avg = average_delivery_time(zone, block, PROMISE)
        pair_averages.append((zone, block, avg))

ranked_avg = sorted(pair_averages, key=lambda pair: pair[2], reverse=True)

print(f"{'Rank':<5} {'Zone':<9} {'Time block':<12} Avg time")
for rank, (zone, block, avg) in enumerate(ranked_avg, start=1):
    print(f"{rank:<5} {zone:<9} {block:<12} {avg:5.1f} min")

Rank  Zone      Time block   Avg time
1     Far West  Fri/Sat eve   42.9 min
2     North     Fri/Sat eve   36.8 min
3     Far West  Weekday eve   35.0 min
4     Central   Fri/Sat eve   32.1 min
5     North     Weekday eve   31.4 min
6     Far West  Lunch         30.6 min
7     Far West  Other         30.0 min
8     Central   Weekday eve   29.0 min
9     North     Lunch         28.7 min
10    North     Other         27.7 min
11    Central   Lunch         26.8 min
12    Central   Other         26.0 min


The busy evening pairs are again at the top: Far West Fri/Sat eve (about 43 minutes), North Fri/Sat eve (about 37) and Far West weekday eve (about 35). But all the averages fall between about 26 and 43 minutes, and every one of them is below 45. Looking only at this table, Rosa might think the promise is being kept everywhere, which part (b) showed is not true.

### (e) Which measure is more relevant: late rate or average delivery time?

To compare the two, the next cell puts the rankings from (b) and (d) side by side, in the order of the average time ranking.

In [9]:
# Look up each pair's late rate and rank from part (b)
late_lookup = {}
for rank, (zone, block, rate) in enumerate(ranked, start=1):
    late_lookup[(zone, block)] = (rank, rate)

print(f"{'Zone':<9} {'Time block':<12} {'Avg time':>9} {'Rank':>5}   {'Late rate':>9} {'Rank':>5}")
for avg_rank, (zone, block, avg) in enumerate(ranked_avg, start=1):
    late_rank, rate = late_lookup[(zone, block)]
    print(f"{zone:<9} {block:<12} {avg:6.1f} min {avg_rank:>5}   {rate:8.1f}% {late_rank:>5}")

Zone      Time block    Avg time  Rank   Late rate  Rank
Far West  Fri/Sat eve    42.9 min     1       36.7%     1
North     Fri/Sat eve    36.8 min     2       16.7%     2
Far West  Weekday eve    35.0 min     3       11.2%     3
Central   Fri/Sat eve    32.1 min     4        6.8%     4
North     Weekday eve    31.4 min     5        6.0%     5
Far West  Lunch          30.6 min     6        2.5%    12
Far West  Other          30.0 min     7        4.7%     6
Central   Weekday eve    29.0 min     8        4.2%     7
North     Lunch          28.7 min     9        3.7%     8
North     Other          27.7 min    10        2.5%    11
Central   Lunch          26.8 min    11        3.5%     9
Central   Other          26.0 min    12        2.6%    10


I think the late rate is more relevant to Rosa's decisions, for these reasons.

First, Rosa's costs come from late orders. Every order that misses the promise gets a refund and the customer orders less in the future. An order delivered in 44 minutes costs nothing extra, but one delivered in 46 minutes does. So what matters is how many orders go over the promise, which is what the late rate measures. The average delivery time doesn't enter the cost at all.

Second, the average hides the problem. All 12 pairs have an average under 45 minutes, yet about 8% of orders are late overall. Far West on Fri/Sat evenings averages 42.9 minutes but 36.7% of its orders are late. North on Fri/Sat evenings averages 36.8 minutes, about 8 minutes under the promise, and still about 1 in 6 orders is late. Fast deliveries pull the average down, but each slow one is still an unhappy customer and a refund.

Third, the late rate shows the size of the problem much more clearly. The averages only range from about 26 to 43 minutes, while the late rates go from 2.5% to 36.7%. Far West Fri/Sat eve is only about 6 minutes slower on average than North Fri/Sat eve, but its late rate is more than twice as high.

Fourth, the rankings disagree in ways that could mislead Rosa. The top five pairs are the same in both, but after that they differ. Far West at lunch is 6th by average time (30.6 minutes) but last by late rate (2.5%), so going by the average Rosa might try to fix a pair that almost never breaks the promise.

Finally, the decision in Part II is about choosing a promise, which works as a cut-off. Changing the promise changes the late rate directly, and through it the refund and churn costs, while the average can't tell her how many orders would land on either side of a new promise.

The average is still a useful general measure of how fast deliveries are, but for choosing the promise and estimating its cost, the late rate is the one to use.

## Part II. Exploring the best promise

### (a) Total cost per late order

A late order costs Rosa in two ways. She refunds the customer, and the customer orders less in the future, so she loses the profit margin on each of those lost orders. The total cost per late order is the refund plus the number of lost orders times the margin per order.

The function takes the costs dictionary as an input, rather than using COSTS directly, so the web app in Part III can pass in different numbers.

In [10]:
def cost_per_late_order(costs):
    """Total cost of one late order: the refund plus the profit lost from future orders."""
    return costs['refund'] + costs['churn_orders'] * costs['margin']


print("COSTS:", COSTS)
print(f"Cost per late order: ${cost_per_late_order(COSTS):.2f}")

COSTS: {'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}
Cost per late order: $26.20


Each late order costs Rosa \$26.20: the \$10 refund plus 1.8 lost orders times the \$9 margin (\$16.20). That is about 2.9 times what she makes on one order, so one late order cancels out the profit from roughly three on-time orders.

### (b) Choosing the best promise

Net profit for a zone and time block at a given promise is the margin from all orders minus the cost of the late ones: orders × margin − late orders × cost per late order.

The promise pulls this in two directions. A shorter promise brings in more orders, but more of them are late. A longer promise means fewer late orders, but fewer customers order at all. The best promise is where these balance out. Since both the number of orders and the delivery times change with the promise, the simulator has to be run again for every promise we try.

The helper function below calculates the net profit for one promise, and also returns the number of orders and late orders.

In [11]:
def net_profit(zone, time_block, promise, costs, seed=1):
    """Net profit for one zone, time block and promise.

    Returns (net_profit, number_of_orders, number_of_late_orders).
    """
    times = delivery_times(zone, time_block, promise, seed=seed)
    orders = len(times)
    late = int(np.sum(times > promise))
    profit = orders * costs['margin'] - late * cost_per_late_order(costs)
    return profit, orders, late

#### Choosing the range of promises

Before writing the search I want to see what a sensible range looks like, so the next cell tries a very wide range, 5 to 120 minutes in steps of 5, for the worst pair from Part I (Far West, Fri/Sat eve).

In [12]:
print(f"Far West, Fri/Sat eve (cost per late order = ${cost_per_late_order(COSTS):.2f})\n")
print(f"{'Promise':>7} {'Orders':>7} {'Late':>5} {'Late %':>7} {'Net profit':>11}")
for promise in range(5, 125, 5):
    profit, orders, late = net_profit("Far West", "Fri/Sat eve", promise, COSTS)
    late_pct = 100 * late / orders if orders > 0 else 0
    print(f"{promise:>7} {orders:>7} {late:>5} {late_pct:>6.1f}% {profit:>11.1f}")

Far West, Fri/Sat eve (cost per late order = $26.20)

Promise  Orders  Late  Late %  Net profit
      5     242   242  100.0%     -4162.4
     10     242   242  100.0%     -4162.4
     15     241   241  100.0%     -4145.2
     20     241   239   99.2%     -4092.8
     25     239   233   97.5%     -3953.6
     30     237   218   92.0%     -3578.6
     35     232   179   77.2%     -2601.8
     40     224   133   59.4%     -1468.6
     45     210    77   36.7%      -127.4
     50     188    33   17.6%       827.4
     55     158     8    5.1%      1212.4
     60     121     2    1.7%      1036.6
     65      84     0    0.0%       756.0
     70      54     0    0.0%       486.0
     75      32     0    0.0%       288.0
     80      18     0    0.0%       162.0
     85      10     0    0.0%        90.0
     90       6     0    0.0%        54.0
     95       3     0    0.0%        27.0
    100       2     0    0.0%        18.0
    105       1     0    0.0%         9.0
    110       0     0 

A few things stand out from this scan:

- At 20 minutes or less almost every order is late, and net profit is around −\$4,000, since each late order costs \$26.20 but only earns \$9. None of these could be the best promise, so the range doesn't need to start lower than about 20.
- From 65 minutes there are no late orders left, so a longer promise only loses customers, and at 110 minutes nobody orders at all. The range only needs to go a bit past the point where late orders reach zero.
- The best promise should be inside the range, not at one end. If it is at an end, the real best might be outside the range, so the range should be widened. The example range in the assignment (5 to 50) shows why this matters: for this pair it would pick 50 minutes, but 55 minutes actually earns more (\$1,212 vs \$827).
- Steps of 5 minutes keep the promises easy to communicate and keep the number of simulations small.

So I use 20 to 100 minutes in steps of 5 for every pair. The search function also prints a warning if the best promise is at either end of the list.

In [13]:
def best_promise(zone, time_block, promises, costs, seed=1):
    """Try every promise in `promises` and return the one with the highest net profit.

    Returns (best_promise, best_net_profit, results), where results is a list of
    (promise, orders, late_orders, net_profit) tuples, one per promise tried.
    """
    promises = sorted(promises)
    results = []
    best_p = None
    best_profit = None
    for promise in promises:
        profit, orders, late = net_profit(zone, time_block, promise, costs, seed=seed)
        results.append((promise, orders, late, profit))
        if best_profit is None or profit > best_profit:
            best_p = promise
            best_profit = profit

    if len(promises) > 1 and best_p in (promises[0], promises[-1]):
        print(f"Warning: the best promise ({best_p} min) is at the edge of the range "
              f"{promises[0]}-{promises[-1]} min; consider widening the range.")

    return best_p, best_profit, results


PROMISES = list(range(20, 105, 5))
print("Promises to try:", PROMISES)

Promises to try: [20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]


The function goes through the promises in order, works out the net profit for each, and keeps the one with the highest profit. If two promises tie, the shorter one is kept. It returns the best promise, its net profit, and the full list of results so the web app can show them. Its inputs are the zone, time block, list of promises and costs, as the question asks, so it can be reused in the app.

Testing it on Far West, Fri/Sat eve with my range, and then with the 5 to 50 range from the assignment for comparison:

In [14]:
zone, block = "Far West", "Fri/Sat eve"

p, profit, results = best_promise(zone, block, PROMISES, COSTS)
print(f"{zone}, {block}: recommended promise = {p} minutes, net profit = ${profit:,.2f}")

profit_45, orders_45, late_45 = net_profit(zone, block, 45, COSTS)
print(f"For comparison, today's 45-minute promise: net profit = ${profit_45:,.2f} "
      f"({orders_45} orders, {late_45} late)")

print("\nWith the assignment's example range of 5 to 50 minutes:")
p_narrow, profit_narrow, _ = best_promise(zone, block, list(range(5, 55, 5)), COSTS)
print(f"Recommended promise = {p_narrow} minutes, net profit = ${profit_narrow:,.2f}")

Far West, Fri/Sat eve: recommended promise = 55 minutes, net profit = $1,212.40
For comparison, today's 45-minute promise: net profit = $-127.40 (210 orders, 77 late)

With the assignment's example range of 5 to 50 minutes:
Recommended promise = 50 minutes, net profit = $827.40


For Far West on Fri/Sat evenings the best promise is 55 minutes, with a net profit of about \$1,212 over four weeks. With the current 45-minute promise this pair actually loses money (about −\$127), because 77 of its 210 orders are late. At 55 minutes there are fewer orders (158 instead of 210), but only 8 are late. The 5 to 50 range picks 50 minutes and shows the edge warning, which confirms that the range needs to go past the best value.

#### Best promise for every zone and time block

Rosa needs a promise for each pair, so I run the same search for all 12 pairs and compare each one with the current 45-minute promise.

In [15]:
total_today = 0
total_best = 0

print(f"{'Zone':<9} {'Time block':<12} {'Best':>5} {'Net profit':>11} {'At 45 min':>10} {'Gain':>9}")
for zone in ZONES:
    for block in TIME_BLOCKS:
        p, profit, _ = best_promise(zone, block, PROMISES, COSTS)
        profit_45, _, _ = net_profit(zone, block, 45, COSTS)
        total_best += profit
        total_today += profit_45
        print(f"{zone:<9} {block:<12} {p:>5} {profit:>11,.1f} {profit_45:>10,.1f} {profit - profit_45:>9,.1f}")

print(f"\nTotal net profit over four weeks: today (45 min everywhere) = ${total_today:,.2f}, "
      f"best promises = ${total_best:,.2f}")
print(f"Gain: ${total_best - total_today:,.2f} ({100 * (total_best - total_today) / total_today:.1f}%)")

Zone      Time block    Best  Net profit  At 45 min      Gain
Central   Lunch           45     2,104.2    2,104.2       0.0
Central   Weekday eve     45     2,603.2    2,603.2       0.0
Central   Fri/Sat eve     50     1,399.0    1,369.4      29.6
Central   Other           45     1,579.0    1,579.0       0.0
North     Lunch           45     1,526.6    1,526.6       0.0
North     Weekday eve     45     1,857.0    1,857.0       0.0
North     Fri/Sat eve     50       996.4      695.0     301.4
North     Other           45     1,335.2    1,335.2       0.0
Far West  Lunch           45     1,001.4    1,001.4       0.0
Far West  Weekday eve     50     1,210.8    1,032.2     178.6
Far West  Fri/Sat eve     55     1,212.4     -127.4   1,339.8
Far West  Other           45     1,492.2    1,492.2       0.0

Total net profit over four weeks: today (45 min everywhere) = $16,468.00, best promises = $18,317.40
Gain: $1,849.40 (11.2%)


My recommendation for Rosa is to keep the 45-minute promise for 8 of the 12 pairs (all lunches, all "Other" times, and weekday evenings in Central and North), since 45 minutes is already the most profitable option there. For the busy evening pairs she should lengthen the promise: 55 minutes for Far West on Fri/Sat evenings, which turns a loss of about \$127 into a profit of about \$1,212, and 50 minutes for North Fri/Sat eve (about \$301 more), Far West weekday eve (about \$179 more) and Central Fri/Sat eve (about \$30 more).

Together this raises the four-week net profit from about \$16,468 to \$18,317, a gain of about \$1,849 or 11%, without changing driver staffing. These are the same pairs that had the highest late rates in Part I. The numbers come from one simulated four-week period (seed=1), so the small gains like Central Fri/Sat eve's \$30 are less certain, but the gain in Far West on weekend evenings is clear.